# Lecture 11

In [2]:
#!/usr/bin/env python3
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

OUT_DIR = "figs"
os.makedirs(OUT_DIR, exist_ok=True)

def savefig(fig, stem):
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, f"{stem}.pdf"), bbox_inches="tight")
    fig.savefig(os.path.join(OUT_DIR, f"{stem}.png"), dpi=220, bbox_inches="tight")
    plt.close(fig)

def plot_relu_regions(stem="relu_piecewise_regions",
                      eps=0.45, r_small=0.20, r_large=0.55,
                      lim=1.2, n=400):
    """
    ReLU example: f(x1,x2)=max(0,x1)-max(0,x2)
    Show 4 regimes, point x=(eps,eps), and two neighborhood radii.
    """
    x1 = np.linspace(-lim, lim, n)
    x2 = np.linspace(-lim, lim, n)
    X1, X2 = np.meshgrid(x1, x2)

    # regime id based on signs: 0..3
    # 0: (+,+), 1:(-,+), 2:(-,-), 3:(+,-)
    reg = np.zeros_like(X1, dtype=int)
    reg[(X1 < 0) & (X2 >= 0)] = 1
    reg[(X1 < 0) & (X2 < 0)]  = 2
    reg[(X1 >= 0) & (X2 < 0)] = 3

    fig, ax = plt.subplots(figsize=(6.5, 5.4))

    # light region coloring using contourf default colormap (no explicit colors)
    ax.contourf(X1, X2, reg, levels=[-0.5,0.5,1.5,2.5,3.5], alpha=0.22, antialiased=True)

    # axes
    ax.axhline(0, linewidth=1.2)
    ax.axvline(0, linewidth=1.2)

    # point x
    ax.plot([eps], [eps], marker="o")
    ax.text(eps + 0.03, eps + 0.02, r"$x$", fontsize=12)

    # neighborhoods
    ax.add_patch(Circle((eps, eps), r_small, fill=False, linewidth=2.0))
    ax.add_patch(Circle((eps, eps), r_large, fill=False, linewidth=2.0, linestyle="--"))

    # labels placed away from center to avoid overlap
    ax.text(0.55*lim, 0.78*lim, r"$x_1>0,\;x_2>0$", fontsize=10, ha="center")
    ax.text(-0.55*lim, 0.78*lim, r"$x_1<0,\;x_2>0$", fontsize=10, ha="center")
    ax.text(-0.55*lim, -0.78*lim, r"$x_1<0,\;x_2<0$", fontsize=10, ha="center")
    ax.text(0.55*lim, -0.78*lim, r"$x_1>0,\;x_2<0$", fontsize=10, ha="center")

    ax.set_title(r"ReLU regimes for $f(x_1,x_2)=\max\{0,x_1\}-\max\{0,x_2\}$")
    ax.set_xlabel(r"$x_1$")
    ax.set_ylabel(r"$x_2$")
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal", adjustable="box")

    savefig(fig, stem)

def plot_sigmoid_soft_boundary(stem="sigmoid_soft_boundary",
                               w=(1.0, -0.55), b=0.05,
                               band=0.35, eps=0.15,
                               r_small=0.18, r_large=0.48,
                               lim=1.2, n=420):
    """
    Sigmoid example: f(x)=sigma(w^T x + b)
    Show:
      - contourf of f (smooth shading)
      - the boundary w^T x + b = 0
      - a transition band |w^T x + b| <= band
      - point x and two radii
    """
    w = np.array(w, dtype=float)
    wnorm = np.linalg.norm(w)
    if wnorm == 0:
        raise ValueError("w must be nonzero")
    w = w / wnorm  # normalize for geometric clarity

    x1 = np.linspace(-lim, lim, n)
    x2 = np.linspace(-lim, lim, n)
    X1, X2 = np.meshgrid(x1, x2)

    T = w[0]*X1 + w[1]*X2 + b
    F = 1.0 / (1.0 + np.exp(-T))  # sigmoid

    fig, ax = plt.subplots(figsize=(6.5, 5.4))

    # smooth background: f values (default colormap, no explicit colors)
    ax.contourf(X1, X2, F, levels=18, alpha=0.25)

    # transition band: mask and overlay
    band_mask = (np.abs(T) <= band).astype(float)
    ax.contourf(X1, X2, band_mask, levels=[0.5, 1.5], alpha=0.18)

    # boundary line: solve w0*x1 + w1*x2 + b = 0
    # param x1 -> x2
    xs = np.linspace(-lim, lim, 400)
    if abs(w[1]) > 1e-9:
        ys = (-w[0]*xs - b) / w[1]
        ax.plot(xs, ys, linewidth=2.0)
    else:
        # vertical line
        x_const = -b / w[0]
        ax.plot([x_const, x_const], [-lim, lim], linewidth=2.0)

    # point x: place it slightly on one side of boundary (controlled by eps along normal)
    # choose a base point near origin projected onto boundary, then shift by eps along normal
    # normal to boundary is w
    x0 = -b * w  # simple point whose dot gives ~ -b (near boundary for normalized w)
    xpt = x0 - eps * w  # shift into one side
    ax.plot([xpt[0]], [xpt[1]], marker="o")
    ax.text(xpt[0] + 0.03, xpt[1] + 0.02, r"$x$", fontsize=12)

    # neighborhoods
    ax.add_patch(Circle((xpt[0], xpt[1]), r_small, fill=False, linewidth=2.0))
    ax.add_patch(Circle((xpt[0], xpt[1]), r_large, fill=False, linewidth=2.0, linestyle="--"))

    # axes
    ax.axhline(0, linewidth=1.2)
    ax.axvline(0, linewidth=1.2)

    ax.set_title(r"Sigmoid soft boundary for $f(x)=\sigma(w^\top x + b)$")
    ax.set_xlabel(r"$x_1$")
    ax.set_ylabel(r"$x_2$")
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal", adjustable="box")

    # small annotation that won't overlap
    ax.text(-1.15, 1.08, r"transition band: $|w^\top x+b|\leq \tau$", fontsize=10)

    savefig(fig, stem)

if __name__ == "__main__":
    plot_relu_regions()
    plot_sigmoid_soft_boundary()
    print(f"Saved figures to: {OUT_DIR}/")

Saved figures to: figs/


In [ ]:
#!/usr/bin/env python3
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

OUT_DIR = "figs"
os.makedirs(OUT_DIR, exist_ok=True)

def save(fig, name):
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, f"{name}.pdf"), bbox_inches="tight")
    fig.savefig(os.path.join(OUT_DIR, f"{name}.png"), dpi=220, bbox_inches="tight")
    plt.close(fig)

def fig_relu_coeff_drift(name="relu_coeff_drift_regions",
                         eps=0.45, r_small=0.20, r_large=0.55,
                         lim=1.2, n=450):
    x1 = np.linspace(-lim, lim, n)
    x2 = np.linspace(-lim, lim, n)
    X1, X2 = np.meshgrid(x1, x2)

    # region id by sign pattern
    reg = np.zeros_like(X1, dtype=int)
    reg[(X1 < 0) & (X2 >= 0)] = 1
    reg[(X1 < 0) & (X2 < 0)]  = 2
    reg[(X1 >= 0) & (X2 < 0)] = 3

    fig, ax = plt.subplots(figsize=(6.6, 5.4))
    ax.contourf(X1, X2, reg, levels=[-0.5,0.5,1.5,2.5,3.5], alpha=0.22)

    ax.axhline(0, linewidth=1.2)
    ax.axvline(0, linewidth=1.2)

    ax.plot([eps], [eps], marker="o")
    ax.text(eps + 0.03, eps + 0.02, r"$x=(\epsilon,\epsilon)$", fontsize=11)

    ax.add_patch(Circle((eps, eps), r_small, fill=False, linewidth=2.2))
    ax.add_patch(Circle((eps, eps), r_large, fill=False, linewidth=2.2, linestyle="--"))

    # quadrant labels (moved outward)
    ax.text( 0.70*lim,  0.86*lim, r"$x_1>0,\;x_2>0$", fontsize=10, ha="center")
    ax.text(-0.70*lim,  0.86*lim, r"$x_1<0,\;x_2>0$", fontsize=10, ha="center")
    ax.text(-0.70*lim, -0.86*lim, r"$x_1<0,\;x_2<0$", fontsize=10, ha="center")
    ax.text( 0.70*lim, -0.86*lim, r"$x_1>0,\;x_2<0$", fontsize=10, ha="center")

    ax.set_xlabel(r"$x_1$")
    ax.set_ylabel(r"$x_2$")
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal", adjustable="box")
    save(fig, name)

def fig_sigmoid_1d_saturation(name="sigmoid_1d_saturation",
                              a=4.0, b=0.0,
                              x_center_transition=0.0,
                              x_center_saturation=1.1,
                              r=0.35):
    def sig(t): return 1.0/(1.0+np.exp(-t))

    xs = np.linspace(-2.2, 2.2, 600)
    f = sig(a*xs + b)

    fig, ax = plt.subplots(figsize=(7.2, 4.0))
    ax.plot(xs, f, linewidth=2.2)

    def tangent_line(x0):
        y0 = sig(a*x0+b)
        fp = a*y0*(1-y0)
        return y0 + fp*(xs-x0), fp

    # transition neighborhood
    ytan_tr, slope_tr = tangent_line(x_center_transition)
    ax.plot(xs, ytan_tr, linestyle="--", linewidth=1.8)
    ax.axvspan(x_center_transition - r, x_center_transition + r, alpha=0.15)
    ax.text(x_center_transition, 0.08,
            rf"transition neighborhood, slope $\approx {slope_tr:.2f}$",
            fontsize=10, ha="center")

    # saturation neighborhood
    ytan_sat, slope_sat = tangent_line(x_center_saturation)
    ax.plot(xs, ytan_sat, linestyle="--", linewidth=1.8)
    ax.axvspan(x_center_saturation - r, x_center_saturation + r, alpha=0.15)
    ax.text(x_center_saturation, 0.92,
            rf"saturation neighborhood, slope $\approx {slope_sat:.2f}$",
            fontsize=10, ha="center")

    ax.set_xlabel(r"$x$")
    ax.set_ylabel(r"$\sigma(ax+b)$")
    ax.set_ylim(-0.05, 1.05)
    save(fig, name)

def fig_sigmoid_soft_boundary(name="sigmoid_soft_boundary",
                              w=(1.0, -0.55), b=0.05,
                              tau=0.25,
                              eps_shift=0.18,
                              r_small=0.18, r_large=0.48,
                              lim=1.2, n=520):
    w = np.array(w, dtype=float)
    wnorm = np.linalg.norm(w)
    if wnorm == 0:
        raise ValueError("w must be nonzero")
    w = w / wnorm

    x1 = np.linspace(-lim, lim, n)
    x2 = np.linspace(-lim, lim, n)
    X1, X2 = np.meshgrid(x1, x2)

    T = w[0]*X1 + w[1]*X2 + b
    F = 1.0/(1.0+np.exp(-T))

    fig, ax = plt.subplots(figsize=(6.6, 5.4))
    ax.contourf(X1, X2, F, levels=20, alpha=0.22)

    # transition band |T| <= tau
    band = (np.abs(T) <= tau).astype(float)
    ax.contourf(X1, X2, band, levels=[0.5, 1.5], alpha=0.18)

    # boundary line T=0
    xs = np.linspace(-lim, lim, 500)
    if abs(w[1]) > 1e-9:
        ys = (-w[0]*xs - b) / w[1]
        ax.plot(xs, ys, linewidth=2.2)
    else:
        x_const = -b / w[0]
        ax.plot([x_const, x_const], [-lim, lim], linewidth=2.2)

    # choose a point near boundary, then shift into one side along normal
    x0 = -b * w
    xpt = x0 - eps_shift * w
    ax.plot([xpt[0]], [xpt[1]], marker="o")
    ax.text(xpt[0] + 0.03, xpt[1] + 0.02, r"$x$", fontsize=12)

    ax.add_patch(Circle((xpt[0], xpt[1]), r_small, fill=False, linewidth=2.2))
    ax.add_patch(Circle((xpt[0], xpt[1]), r_large, fill=False, linewidth=2.2, linestyle="--"))

    ax.axhline(0, linewidth=1.2)
    ax.axvline(0, linewidth=1.2)

    # Use \leq (mathtext-safe) not \le
    ax.text(-1.15, 1.08, r"transition band: $|w^\top x+b|\leq \tau$", fontsize=10)

    ax.set_xlabel(r"$x_1$")
    ax.set_ylabel(r"$x_2$")
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal", adjustable="box")
    save(fig, name)

if __name__ == "__main__":
    fig_relu_coeff_drift()
    fig_sigmoid_1d_saturation()
    fig_sigmoid_soft_boundary()
    print(f"Saved PDFs/PNGs to {OUT_DIR}/")

Saved PDFs/PNGs to figs/


In [6]:
#!/usr/bin/env python3
"""
Two-panel (PDF only): Linear vs Sigmoid to illustrate local surrogate behavior vs r.

Fixes:
  - Numerically stable weights (prevents underflow => N_eff=0).
  - Uses a truly nonlinear model (sigmoid) so bias increases with r.
  - Linear panel is expected to have ~0 bias and ~0 variance (feature, not bug).

Output:
  figs/local_surrogate_linear_vs_sigmoid.pdf
"""

import os
import numpy as np
import matplotlib.pyplot as plt

OUT_DIR = "figs"
os.makedirs(OUT_DIR, exist_ok=True)

def sigmoid(t):
    return 1.0 / (1.0 + np.exp(-t))

# --------- models ---------

def f_linear(Z, w):
    return Z @ w

def grad_linear(x, w):
    return w.copy()

def f_sigmoid(Z, w, b):
    return sigmoid(Z @ w + b)

def grad_sigmoid(x, w, b):
    t = float(x @ w + b)
    s = float(sigmoid(np.array(t)))
    sp = s * (1.0 - s)
    return sp * w

# --------- numerically stable weights ---------

def stable_rbf_weights(dist2, r):
    """
    w_i = exp(-dist2/(2 r^2)), stabilized to avoid underflow:
      a = -dist2/(2 r^2)
      a <- a - max(a)
      w = exp(a)
    """
    a = -dist2 / (2.0 * (r * r) + 1e-30)
    a = a - np.max(a)  # stabilization
    w = np.exp(a)
    # If everything is zero (shouldn't happen now), fall back safely
    if not np.isfinite(w).all() or w.sum() == 0.0:
        w = np.ones_like(w)
    return w

# --------- weighted local linear surrogate (centered) ---------

def weighted_ls_beta(delta, y, w, solve_ridge=1e-12):
    """
    Fit y ≈ beta0 + beta^T delta with weights w.

    delta: (N,2)  where delta = Z - x
    y:     (N,)
    w:     (N,)

    Returns: beta (2,), kappa, neff
    """
    N = delta.shape[0]
    X = np.concatenate([np.ones((N, 1), dtype=float), delta], axis=1)  # (N,3)

    XtWX = (X.T * w) @ X
    XtWy = (X.T * w) @ y

    # effective sample size
    neff = (w.sum() ** 2) / (np.sum(w * w) + 1e-30)

    # conditioning (depends only on X and w)
    try:
        kappa = float(np.linalg.cond(XtWX))
    except np.linalg.LinAlgError:
        kappa = np.inf

    beta_full = np.linalg.solve(XtWX + solve_ridge * np.eye(3), XtWy)
    return beta_full[1:], kappa, neff

def run_curve(
    *,
    f_eval,
    grad_eval,
    x,
    rs,
    REPS=20,
    N=6000,
    sample_scale=0.35,
    solve_ridge=1e-12,
    seed=0,
):
    """
    Returns dict with arrays: bias, var, log10kappa, neff
    """
    rng = np.random.default_rng(seed)
    grad_true = grad_eval(x)

    betas_by_r = [ [] for _ in rs ]
    kappas_by_r = [ [] for _ in rs ]
    neff_by_r = [ [] for _ in rs ]

    for _ in range(REPS):
        delta = rng.normal(scale=sample_scale, size=(N, 2))
        Z = x[None, :] + delta
        y = f_eval(Z)

        dist2 = np.sum(delta * delta, axis=1)

        for j, r in enumerate(rs):
            w = stable_rbf_weights(dist2, float(r))
            beta, kappa, neff = weighted_ls_beta(delta, y, w, solve_ridge=solve_ridge)
            betas_by_r[j].append(beta)
            kappas_by_r[j].append(kappa)
            neff_by_r[j].append(neff)

    bias_curve = []
    var_curve = []
    kappa_curve = []
    neff_curve = []

    for j in range(len(rs)):
        B = np.vstack(betas_by_r[j])  # (REPS,2)
        b_mean = B.mean(axis=0)
        b_std = B.std(axis=0)

        bias_curve.append(float(np.linalg.norm(b_mean - grad_true)))
        var_curve.append(float(np.linalg.norm(b_std)))
        kappa_curve.append(float(np.median(kappas_by_r[j])))
        neff_curve.append(float(np.median(neff_by_r[j])))

    kappa_curve = np.array(kappa_curve, dtype=float)
    # avoid log10(inf) warnings in plot by clipping at a large number
    kappa_clip = np.minimum(kappa_curve, 1e16)

    return {
        "rs": np.array(rs, dtype=float),
        "bias": np.array(bias_curve, dtype=float),
        "var": np.array(var_curve, dtype=float),
        "log10kappa": np.log10(kappa_clip),
        "neff": np.array(neff_curve, dtype=float),
    }

def main():
    x = np.array([0.30, -0.25], dtype=float)

    # IMPORTANT: don’t include absurdly tiny r compared to sample_scale,
    # but stabilized weights now prevent outright zeros anyway.
    rs = np.logspace(-1.3, 0.2, 18)  # ~0.05 to ~1.58

    REPS = 20
    N = 6000
    sample_scale = 0.35
    solve_ridge = 1e-12

    w = np.array([2.0, -1.0], dtype=float)
    b = -0.2

    lin = run_curve(
        f_eval=lambda Z: f_linear(Z, w),
        grad_eval=lambda x0: grad_linear(x0, w),
        x=x, rs=rs,
        REPS=REPS, N=N,
        sample_scale=sample_scale,
        solve_ridge=solve_ridge,
        seed=0,
    )

    sig = run_curve(
        f_eval=lambda Z: f_sigmoid(Z, w, b),
        grad_eval=lambda x0: grad_sigmoid(x0, w, b),
        x=x, rs=rs,
        REPS=REPS, N=N,
        sample_scale=sample_scale,
        solve_ridge=solve_ridge,
        seed=1,
    )

    fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.6), sharex=True)

    def panel(ax, data, title):
        r = data["rs"]
        ax.plot(r, data["bias"], marker="o", label="bias")
        ax.plot(r, data["var"], marker="o", label="variance")
        ax.plot(r, data["log10kappa"], marker="o", label=r"$\log_{10}\kappa$")

        # rescale N_eff to sit on same y-axis (purely for visibility)
        neff = data["neff"]
        scale = (np.max(data["bias"]) + 1e-12)
        neff_rescaled = neff / (np.max(neff) + 1e-30) * scale
        ax.plot(r, neff_rescaled, marker="o",
                label=r"$N_{\mathrm{eff}}$ (rescaled)")

        ax.set_xscale("log")
        ax.set_xlabel(r"kernel bandwidth $r$")
        ax.set_title(title)
        ax.grid(True, which="both", linewidth=0.6, alpha=0.4)

    panel(axes[0], lin, r"Linear: $f(x)=w^\top x$")
    panel(axes[1], sig, r"Sigmoid: $f(x)=\sigma(w^\top x + b)$")

    axes[0].set_ylabel("magnitude / proxy")
    axes[0].legend(frameon=True, fontsize=9, loc="best")

    fig.suptitle("Local linear surrogate vs neighborhood scale", y=1.02)
    fig.tight_layout()

    out_pdf = os.path.join(OUT_DIR, "local_surrogate_linear_vs_sigmoid.pdf")
    fig.savefig(out_pdf, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", out_pdf)

if __name__ == "__main__":
    main()

Saved: figs/local_surrogate_linear_vs_sigmoid.pdf


In [12]:
#!/usr/bin/env python3
"""
1x3 (PDF only): Local surrogate diagnostics vs neighborhood scale.

Panels:
  (1) Bias curves: linear + sigmoid (same axes)
  (2) log10 kappa(X^T W X)
  (3) N_eff

Notes:
  - kappa and N_eff depend only on (X, W): perturbation geometry + kernel weights.
    They do NOT depend on the model f, so we plot them once.
  - Bias depends on the model, so we overlay both biases in panel (1).

Style:
  - same marker ("o") everywhere
  - default Matplotlib colors (C0 blue, C1 orange, C2 green)

Output:
  figs/local_surrogate_1x3_bias_kappa_neff.pdf
"""

import os
import numpy as np
import matplotlib.pyplot as plt

OUT_DIR = "figs"
os.makedirs(OUT_DIR, exist_ok=True)

def sigmoid(t):
    return 1.0 / (1.0 + np.exp(-t))

# ----------------------------
# Models
# ----------------------------

def f_linear(Z, w):
    return Z @ w

def grad_linear(x, w):
    return w.copy()

def f_sigmoid(Z, w, b):
    return sigmoid(Z @ w + b)

def grad_sigmoid(x, w, b):
    t = float(x @ w + b)
    s = float(sigmoid(np.array(t)))
    sp = s * (1.0 - s)
    return sp * w

# ----------------------------
# Stable RBF weights
# ----------------------------

def stable_rbf_weights(dist2, r):
    a = -dist2 / (2.0 * (r * r) + 1e-30)
    a = a - np.max(a)
    w = np.exp(a)
    if w.sum() == 0.0 or not np.isfinite(w).all():
        w = np.ones_like(w)
    return w

# ----------------------------
# Weighted local linear surrogate (centered)
# y ≈ beta0 + beta^T delta
# ----------------------------

def weighted_ls_beta(delta, y, w, solve_ridge=1e-12):
    N = delta.shape[0]
    X = np.concatenate([np.ones((N, 1)), delta], axis=1)  # (N,3)

    XtWX = (X.T * w) @ X
    XtWy = (X.T * w) @ y

    neff = (w.sum() ** 2) / (np.sum(w * w) + 1e-30)

    try:
        kappa = float(np.linalg.cond(XtWX))
    except np.linalg.LinAlgError:
        kappa = np.inf

    beta_full = np.linalg.solve(XtWX + solve_ridge * np.eye(3), XtWy)
    return beta_full[1:], kappa, neff

# ----------------------------
# Shared geometry perturbations (anisotropic, near 1D)
# ----------------------------

def sample_anisotropic_delta(rng, N, major=1.0, minor=0.006):
    u = rng.normal(size=(N, 1))
    v = rng.normal(size=(N, 1))
    return np.hstack([major * u, minor * v])

def main():
    # r sweep (small -> moderate)
    rs = np.logspace(-2.6, 0.2, 24)

    # geometry knobs
    major = 1.0
    minor = 0.006  # smaller => more extreme kappa at small r

    # runtime knobs
    REPS = 10
    N = 7000
    solve_ridge = 1e-12

    # shared x for both models
    x = np.array([0.02, -0.01], dtype=float)

    # linear model
    w_lin = np.array([1.3, -0.7], dtype=float)

    # sigmoid model (steep, near boundary)
    w_sig = np.array([5.0, -4.0], dtype=float)
    b_sig = 0.0

    grad_true_lin = grad_linear(x, w_lin)
    grad_true_sig = grad_sigmoid(x, w_sig, b_sig)

    rng = np.random.default_rng(0)

    # pre-sample shared delta clouds
    deltas = []
    dist2s = []
    for _ in range(REPS):
        delta = sample_anisotropic_delta(rng, N, major=major, minor=minor)
        deltas.append(delta)
        dist2s.append(np.sum(delta * delta, axis=1))

    betas_lin = [[] for _ in rs]
    betas_sig = [[] for _ in rs]
    kappas = [[] for _ in rs]
    neffs  = [[] for _ in rs]

    for rep in range(REPS):
        delta = deltas[rep]
        dist2 = dist2s[rep]
        Z = x[None, :] + delta

        y_lin = f_linear(Z, w_lin)
        y_sig = f_sigmoid(Z, w_sig, b_sig)

        for j, r in enumerate(rs):
            w = stable_rbf_weights(dist2, float(r))

            beta_l, kappa, neff = weighted_ls_beta(delta, y_lin, w, solve_ridge=solve_ridge)
            beta_s, _, _        = weighted_ls_beta(delta, y_sig, w, solve_ridge=solve_ridge)

            betas_lin[j].append(beta_l)
            betas_sig[j].append(beta_s)
            kappas[j].append(kappa)
            neffs[j].append(neff)

    def reduce_bias(betas_by_r, grad_true):
        bias = []
        for j in range(len(rs)):
            B = np.vstack(betas_by_r[j])  # (REPS,2)
            b_mean = B.mean(axis=0)
            bias.append(float(np.linalg.norm(b_mean - grad_true)))
        return np.array(bias, dtype=float)

    bias_lin = reduce_bias(betas_lin, grad_true_lin)
    bias_sig = reduce_bias(betas_sig, grad_true_sig)

    kappa_med = np.array([float(np.median(ks)) for ks in kappas], dtype=float)
    kappa_med = np.minimum(kappa_med, 1e20)
    log10kappa = np.log10(kappa_med)

    neff_med = np.array([float(np.median(ns)) for ns in neffs], dtype=float)

    # ----------------------------
    # Plot: 1x3
    # ----------------------------

    fig, ax = plt.subplots(1, 3, figsize=(13.2, 4.0), sharex=True)

    m = "o"

    # (1) Bias (both)
    ax[0].plot(rs, bias_lin, marker=m, label="bias (linear)", color="C0")
    ax[0].plot(rs, bias_sig, marker=m, label="bias (sigmoid)", color="C1")
    ax[0].set_title("bias")
    ax[0].set_ylabel("bias (proxy)")
    ax[0].legend(frameon=True)

    # (2) log10 kappa
    ax[1].plot(rs, log10kappa, marker=m, label=r"$\log_{10}\kappa(X^\top W X)$", color="C0")
    ax[1].set_title(r"$\log_{10}\kappa(X^\top W X)$")
    ax[1].set_ylabel(r"$\log_{10}\kappa$")
    ax[1].legend(frameon=True)

    # (3) N_eff
    ax[2].plot(rs, neff_med, marker=m, label=r"$N_{\mathrm{eff}}$", color="C0")
    ax[2].set_title(r"$N_{\mathrm{eff}}$")
    ax[2].set_ylabel(r"$N_{\mathrm{eff}}$")
    ax[2].legend(frameon=True)

    for j in range(3):
        ax[j].set_xscale("log")
        ax[j].set_xlabel(r"kernel bandwidth $r$")
        ax[j].grid(True, which="both", alpha=0.4)

    fig.suptitle("Local surrogate diagnostics vs neighborhood scale (1×3)", y=1.04)
    fig.tight_layout()

    out_pdf = os.path.join(OUT_DIR, "local_surrogate_1x3_bias_kappa_neff.pdf")
    fig.savefig(out_pdf, bbox_inches="tight")
    plt.close(fig)

    print("Saved:", out_pdf)
    print("Sanity check (r small -> large):")
    print("  N_eff first,last:", neff_med[0], neff_med[-1])
    print("  log10kappa first,last:", log10kappa[0], log10kappa[-1])
    print("  bias_lin first,last:", bias_lin[0], bias_lin[-1])
    print("  bias_sig first,last:", bias_sig[0], bias_sig[-1])

if __name__ == "__main__":
    main()

Saved: figs/local_surrogate_1x3_bias_kappa_neff.pdf
Sanity check (r small -> large):
  N_eff first,last: 13.50383517192222 6709.312091179726
  log10kappa first,last: 5.383218032366393 4.445285283832904
  bias_lin first,last: 3.4498155006910514e-08 3.2922553572234392e-12
  bias_sig first,last: 0.00018623853493438994 1.0906277920461334
